# Superspreading Signature Detection

This notebook detects node-level superspreading signatures in a temporal cluster-transition network. Each node is a cluster of genetically close and temporally proximate SARS-CoV-2 sequences, defined in overlapping 3-week windows stepped forward by 1 week. Directed edges link clusters in adjacent windows and are weighted by the number of sequences shared between the source and target clusters.

The goal is not to prove that a specific exposure event occurred. Instead, the notebook labels network nodes whose local growth, onward continuity, downstream branching, geographic breadth, or demographic mixing look unusual relative to other nodes observed in the same time window.


## Method Rationale

The overlapping-window design intentionally allows the same sequence to appear in consecutive windows. When a sequence is present in two adjacent windows and its assigned cluster changes, that shared sequence provides evidence of continuity from one node to the next. Edge weights therefore measure observed overlap, not direct transmission counts.

The signature detection uses three complementary dimensions:

1. **Local amplification:** a node is large, novel relative to incoming overlap, or has high net amplification within its window or lifecycle stratum.
2. **Onward dissemination:** a node passes many shared sequences forward, branches into multiple successor nodes, or has an unusually even downstream distribution.
3. **Population mixing and spatial breadth:** a node contains more sex, age, SIMD, or health-board diversity than expected for its size and window, or spans unusually many data zones.

Censoring matters. Nodes at the beginning or end of the full dataset, or at the first/last window of a VOC epoch, have uncertain origins or onward spread. The categories retain those nodes but add a censoring note so they can be filtered during sensitivity checks.


## Setup

The notebook is designed to run from either the repository root or the `sse_detection/` folder. SSE-pipeline statistical helpers live in `sse_detection/lib/stats.py` (general helpers in `utils/stats.py`); notebook cells focus on orchestration and interpretation.


In [ ]:
from collections import defaultdict
from pathlib import Path
import sys

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import data as ld  # noqa: E402
from sse_detection.lib import (  # noqa: E402
    add_sse_node_metrics,
    cluster_socio_demo_entropy,
    categorise_sse_nodes,
    downstream_edge_entropy,
    flag_sse,
    frequencies,
    safe_mode
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

RANDOM_SEED = 42
N_ENTROPY_DRAWS = 1000

## Load Sequence-Level Analysis Data

We read only the columns needed for node summaries, network construction, mixing scores, and contextual interpretation. The loader also attaches policy-period labels so candidate signatures can be reviewed against public-health context.


In [ ]:
columns = [
    "window_id",
    "window_idx",
    "wn_mid_date",
    "cluster_id",
    "cluster_size",
    "cluster_n_datazones",
    "cluster_duration_days",
    "dz_urban_rural_class",
    "dz_health_board",
    "dz_local_authority",
    "datazone",
    "age_band",
    "sex",
    "is_female",
    "dz_simd_quintile",
    "who_voc",
    "clade",
    "dz_7d_test_positivity",
    "wn_positive_tests",
    "wn_prop_sequenced",
    "dz_cum_sequences",
    "dz_cum_positive_tests",
    "dz_cum_prop_sequenced",
    "dz_cum_incidence_per_capita",
]

df_raw = ld.load_analysis_columns_pandas(columns, add_policy=True)

# increase window steps to 2 weeks.
df_filtered = df_raw[df_raw["window_idx"] % 2 == 1].copy()
old_to_new = {
    old: new + 1
    for new, old in enumerate(sorted(df_filtered["window_idx"].unique()))
}

df_filtered["window_idx"] = df_filtered["window_idx"].map(old_to_new)
df_filtered["window_id"] = df_filtered["window_idx"].apply(lambda x: f"W{x:03d}")

df_filtered = df_filtered.reset_index(drop=True)
df_filtered = df_filtered.reset_index(drop=True)

print(f"Sequence-window rows: {len(df_filtered):,}")
print(f"Unique sequences: {df_filtered['sequence_id'].nunique():,}")
print(f"Unique node-clusters: {df_filtered['cluster_id'].nunique():,}")
print(f"Windows: {df_filtered['window_idx'].min()} to {df_filtered['window_idx'].max()}")

display(df_filtered.head())

## Null-Corrected Mixing Scores

Large clusters naturally contain more categories than small clusters. To avoid mistaking size for diversity, each entropy score is compared with a within-window null distribution: for each cluster size, categories are randomly drawn from the category frequencies observed in that same window. Positive z-scores indicate more mixing than expected for the cluster size and window.


In [ ]:
ENTROPY_SPECS = [
    ("sex", "sex"),
    ("age_band", "age"),
    ("dz_simd_quintile", "simd"),
    ("dz_health_board", "health_board"),
    ("dz_urban_rural_class", "urban_rural"),
]

for category_col, prefix in ENTROPY_SPECS:
    df_filtered = cluster_socio_demo_entropy(
        df=df_filtered,
        cluster_col="cluster_id",
        category_col=category_col,
        window_col="window_id",
        n_random=N_ENTROPY_DRAWS,
        random_state=RANDOM_SEED,
        prefix=prefix,
    )

entropy_cols = [col for col in df_filtered.columns if "entropy" in col]
print(f"Attached entropy columns: {len(entropy_cols)}")

## Collapse Rows to Node-Level Cluster Summaries

The raw analysis table is sequence-level, with one row per sequence-window observation. The network analysis needs one row per node-cluster. Summary fields use first/min/max values where the value is node-defined, and counts or modal values where the value is sequence-defined.


In [ ]:
agg = {
    "cluster_size": ("cluster_size", "first"),
    "duration_days": ("cluster_duration_days", "first"),
    "first_collection_date": ("collection_date", "min"),
    "last_collection_date": ("collection_date", "max"),
    
    "who_voc": ("who_voc", "first"),
    "clade": ("clade", "first"),
    "pango_lineage": ("pango_lineage", "first"),
    
    "policy_period": ("policy_period", safe_mode),
    "n_policy_periods": ("policy_period", "nunique"),
    "top_policy_periods": ("policy_period", frequencies),

    "cluster_n_datazones": ("cluster_n_datazones", "first"),
    "dz_urban_rural_class": ("dz_urban_rural_class", safe_mode),
    "urban_rural_class_counts": ("dz_urban_rural_class", frequencies),
    "health_board": ("dz_health_board", safe_mode),
    "n_health_boards": ("dz_health_board", "nunique"),
    "health_board_counts": ("dz_health_board", frequencies),
    "local_authority": ("dz_local_authority", safe_mode),
    "n_local_authorities": ("dz_local_authority", "nunique"),
    "local_authority_counts": ("dz_local_authority", frequencies),

    "n_age_bands": ("age_band", "nunique"),
    "age_band_counts": ("age_band", frequencies),

    "fraction_female": ("is_female", "mean"),
    "sex_counts": ("sex", frequencies),
    
    "simd_quintile": ("dz_simd_quintile", safe_mode),
    "n_simd_quintiles": ("dz_simd_quintile", "nunique"),
    "simd_quintile_counts": ("dz_simd_quintile", frequencies),
    
    "dz_7d_test_positivity": ("dz_7d_test_positivity", "mean"),
    "dz_cum_sequences": ("dz_cum_sequences", "mean"),
    "dz_cum_incidence_per_capita": ("dz_cum_incidence_per_capita", "mean"),
    "dz_cum_positive_tests": ("dz_cum_positive_tests", "mean"),
    "dz_cum_prop_sequenced": ("dz_cum_prop_sequenced", "mean"),
    
    "wn_positive_tests": ("wn_positive_tests", "first"),
    "wn_prop_sequenced": ("wn_prop_sequenced", "first"),
    
    **{col: (col, "first") for col in entropy_cols},
}

cluster_table = (
    df_filtered.groupby(["cluster_id", "window_id", "window_idx", "wn_mid_date"], as_index=False)
    .agg(**agg)
    .sort_values(["window_idx", "cluster_size"], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"Node-clusters: {len(cluster_table):,}")
display(cluster_table.head())

## Build the Adjacent-Window Transition Network

For each sequence, we inspect the ordered set of node-clusters it occupies across windows. A directed edge is added only when two observations are in adjacent windows (`delta == 1`) and the cluster assignment changes. The edge weight is the number of unique sequences supporting that transition.

This produces a conservative continuity graph: it does not infer transmission between all genetically similar clusters; it only records observed overlap in adjacent sliding windows.


In [ ]:
seq_nodes = (
    df_filtered[["sequence_id", "cluster_id", "window_id", "window_idx"]]
    .drop_duplicates()
    .sort_values(["sequence_id", "window_idx"])
)

edge_map: dict[tuple[str, str, object, object, int, int], set[str]] = defaultdict(set)

for sequence_id, group in seq_nodes.groupby("sequence_id", sort=False):
    records = list(
        group[["cluster_id", "window_id", "window_idx"]]
        .itertuples(index=False, name=None)
    )

    for i, (source_node, source_window, source_idx) in enumerate(records):
        for target_node, target_window, target_idx in records[i + 1:]:
            delta = target_idx - source_idx

            if delta == 1 and source_node != target_node:
                edge_key = (
                    source_node,
                    target_node,
                    source_window,
                    target_window,
                    source_idx,
                    target_idx,
                )
                edge_map[edge_key].add(str(sequence_id))
            elif delta > 1:
                break

edge_rows = [
    {
        "source": source,
        "target": target,
        "source_window_id": source_window,
        "target_window_id": target_window,
        "source_window_idx": source_idx,
        "target_window_idx": target_idx,
        "n_shared_sequences": len(shared),
    }
    for (
        source,
        target,
        source_window,
        target_window,
        source_idx,
        target_idx,
    ), shared in edge_map.items()
]

edge_table = pd.DataFrame(
    edge_rows,
    columns=[
        "source",
        "target",
        "source_window_id",
        "target_window_id",
        "source_window_idx",
        "target_window_idx",
        "n_shared_sequences",
    ],
)

if not edge_table.empty:
    edge_table = edge_table.sort_values(
        [
            "source_window_idx",
            "target_window_idx",
            "n_shared_sequences",
            "source",
            "target",
        ],
        ascending=[True, True, False, True, True],
        ignore_index=True,
    )

G_raw = nx.DiGraph()
G_raw.add_nodes_from(cluster_table["cluster_id"])
for row in edge_table.itertuples(index=False):
    G_raw.add_edge(row.source, row.target, weight=row.n_shared_sequences)

components = sorted(
    nx.weakly_connected_components(G_raw),
    key=lambda nodes: (-len(nodes), sorted(nodes)[0]),
)
component_map = {
    node: f"AM{component_idx:05d}"
    for component_idx, nodes in enumerate(components, start=1)
    for node in nodes
}
cluster_table["meta_cluster_id"] = cluster_table["cluster_id"].map(component_map)

print(f"Graph nodes: {G_raw.number_of_nodes():,}")
print(f"Graph edges: {G_raw.number_of_edges():,}")
print(f"Weakly connected components: {len(components):,}")
display(edge_table.head())

## Summarise Meta-Clusters

Weakly connected components are treated as meta-clusters: groups of node-clusters linked by at least one path through adjacent-window overlap. These summaries help distinguish short isolated bursts from persistent lineages that move through many windows.


In [ ]:
node_summary = (
    cluster_table.groupby("meta_cluster_id", as_index=False)
    .agg(
        n_clusters=("cluster_id", "nunique"),
        n_windows=("window_id", "nunique"),
        first_window_mid_date=("wn_mid_date", "min"),
        last_window_mid_date=("wn_mid_date", "max"),
        max_cluster_size=("cluster_size", "max"),
        max_cluster_n_datazones=("cluster_n_datazones", "max"),
    )
)

seq_meta_long = df_filtered[[
    "sequence_id",
    "collection_date",
    "cluster_id",
    "pango_lineage",
    "who_voc",
    "clade",
]].merge(
    cluster_table[["cluster_id", "meta_cluster_id"]].drop_duplicates(),
    on="cluster_id",
    how="left",
)

seq_meta_long["n_candidate_meta_clusters"] = (
    seq_meta_long.groupby("sequence_id")["meta_cluster_id"].transform("nunique")
)
seq_meta_long["ambiguous_meta_assignment"] = seq_meta_long["n_candidate_meta_clusters"] > 1

sequence_summary = (
    seq_meta_long.groupby("meta_cluster_id", as_index=False)
    .agg(
        n_sequences=("sequence_id", "nunique"),
        first_collection_date=("collection_date", "min"),
        last_collection_date=("collection_date", "max"),
        pango_lineage=("pango_lineage", "first"),
        clade=("clade", "first"),
        who_voc=("who_voc", "first"),
        ambiguous_sequence_assignments=("ambiguous_meta_assignment", "sum"),
    )
)

meta_summary = node_summary.merge(sequence_summary, on="meta_cluster_id", how="left")
meta_summary["duration_days"] = (
    meta_summary["last_collection_date"] - meta_summary["first_collection_date"]
).dt.days
meta_summary = meta_summary.sort_values(
    ["n_sequences", "n_clusters"],
    ascending=[False, False],
).reset_index(drop=True)

print(f"Ambiguous sequence-window assignments: {seq_meta_long['ambiguous_meta_assignment'].sum():,}")
display(meta_summary.head(20))

## Weekly Meta-Cluster Growth Check

The graph categories below are node-level signatures. As a complementary check, `flag_sse` computes weekly growth within each meta-cluster using the normalised change `new_sequences / sqrt(previous cumulative size)`. This is useful for identifying weeks where a connected component grows abruptly, even if the node-level graph category is ambiguous.


In [ ]:
weekly_growth = flag_sse(seq_meta_long)
weekly_sse_weeks = weekly_growth.loc[weekly_growth["is_sse"]].sort_values(
    ["norm_change", "new_sequences"],
    ascending=[False, False],
)

print(f"Weekly growth rows: {len(weekly_growth):,}")
print(f"Flagged weekly SSE-like growth rows: {len(weekly_sse_weeks):,}")
display(weekly_sse_weeks.head(20))

## Score Node-Level Signatures

Incoming edge metrics describe how much observed sequence overlap a node inherited from previous windows. Downstream entropy describes whether outgoing overlap is concentrated into one successor or distributed across multiple successors. A permutation null is used for downstream entropy because high or low evenness can arise from the observed edge-weight distribution itself.


In [ ]:
in_metrics = (
    edge_table.groupby("target")
    .agg(
        in_degree=("target", "size"),
        in_strength=("n_shared_sequences", "sum"),
    )
    .reset_index()
    .rename(columns={"target": "cluster_id"})
)

node_stats = cluster_table.merge(in_metrics, on="cluster_id", how="left")

downstream_null = downstream_edge_entropy(
    edge_df=edge_table,
    source_col="source",
    weight_col="n_shared_sequences",
)

node_stats = node_stats.merge(
    downstream_null.reset_index().rename(columns={"source": "cluster_id"}),
    on="cluster_id",
    how="left",
)

node_stats = add_sse_node_metrics(node_stats)
node_stats = categorise_sse_nodes(node_stats)

print(f"SSE-like candidate nodes: {node_stats['sse_candidate'].sum():,} / {len(node_stats):,}")

## Category Review

The final category has two parts: a role (`putative_birth`, `relay_amplifier`, `merged_relay`, `terminal_sink`, `isolated_burst`, or `unclear_origin`) and an onward dynamic (`contained_burst`, `single_dominant_chain`, `multi_branch_seeder`, `multi_branch_expander`, `diffuse_spatial_broadcaster`, `high_volume_onward_spread`, `no_observed_onward_spread`, or `weak_or_ambiguous_onward_spread`).

Rows labelled `not_sse_like` did not pass the amplification screen. Censoring notes should be used when interpreting origin or terminal categories.


In [ ]:
category_counts = (
    node_stats["sse_category"]
    .value_counts(dropna=False)
    .rename_axis("sse_category")
    .reset_index(name="n_nodes")
)
category_counts["pct_nodes"] = category_counts["n_nodes"] / len(node_stats)

display(category_counts)

In [ ]:
role_dynamic_counts = (
    node_stats.loc[node_stats["sse_candidate"]]
    .groupby(["sse_role", "sse_onward_dynamic", "sse_censoring_note"], dropna=False)
    .size()
    .reset_index(name="n_nodes")
    .sort_values("n_nodes", ascending=False)
)

display(role_dynamic_counts)

In [ ]:
review_cols = [
    "cluster_id",
    "meta_cluster_id",
    "window_idx",
    "wn_mid_date",
    "sse_category",
    "sse_censoring_note",
    "cluster_size",
    "in_strength",
    "out_strength",
    "out_degree",
    "dominant_successor_frac",
    "downstream_entropy_norm",
    "core_amplification_score",
    "onward_dissemination_score",
    "mixing_score",
    "cluster_n_datazones",
    "n_health_boards",
    "n_local_authorities",
    "who_voc",
    "clade",
    "pango_lineage",
    "policy_period",
]
review_cols = [col for col in review_cols if col in node_stats.columns]

candidate_review = (
    node_stats.loc[node_stats["sse_candidate"], review_cols]
    .sort_values(
        ["core_amplification_score", "onward_dissemination_score", "cluster_size"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

display(candidate_review.head(30))

## Quick Visual Diagnostic

This plot is deliberately simple: it checks whether the category distribution is dominated by one noisy class and highlights the most common candidate signatures for manual review.


In [ ]:
plot_counts = category_counts.query("sse_category != 'not_sse_like'")

fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(plot_counts))))
ax.barh(plot_counts["sse_category"], plot_counts["n_nodes"], color="#3B82F6")
ax.invert_yaxis()
ax.set_xlabel("Node count")
ax.set_ylabel("")
ax.set_title("Most common SSE-like node categories")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## Objects Produced

The main in-memory outputs are:

- `cluster_table`: one row per node-cluster.
- `edge_table`: directed adjacent-window transition edges with shared-sequence weights.
- `meta_summary`: weakly connected component summaries.
- `weekly_growth`: weekly meta-cluster growth diagnostics.
- `node_stats`: node-level metrics, SSE candidate flags, categories, and censoring notes.
- `candidate_review`: ranked candidate node table for inspection.
- `G_raw`: directed adjacent-window cluster transition network.

The companion file `sse_categorise.md` defines the category labels and their intended interpretation.


In [ ]:
import gzip
import pickle

OUTPUT_DIR = PROJECT_ROOT / "sse_detection" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tables_to_save = {
    "cluster_table": cluster_table,
    "edge_table": edge_table,
    "meta_summary": meta_summary,
    "weekly_growth": weekly_growth,
    "node_stats": node_stats,
    "candidate_review": candidate_review,
}

saved_outputs = []
for name, table in tables_to_save.items():
    path = OUTPUT_DIR / f"{name}.parquet"
    table.to_parquet(path, index=False)
    saved_outputs.append(
        {
            "object": name,
            "format": "parquet",
            "n_rows": len(table),
            "n_columns": len(table.columns),
            "n_nodes": None,
            "n_edges": None,
            "path": str(path.relative_to(PROJECT_ROOT)),
        }
    )

sse_network = G_raw.copy()
nx.set_node_attributes(sse_network, node_stats.set_index("cluster_id").to_dict("index"))

edge_attr_cols = [col for col in edge_table.columns if col not in {"source", "target"}]
edge_attrs = {
    (row["source"], row["target"]): {col: row[col] for col in edge_attr_cols}
    for row in edge_table.to_dict("records")
}
nx.set_edge_attributes(sse_network, edge_attrs)

network_path = OUTPUT_DIR / "sse_transition_network.gpickle.gz"
with gzip.open(network_path, "wb") as f:
    pickle.dump(sse_network, f, protocol=pickle.HIGHEST_PROTOCOL)

saved_outputs.append(
    {
        "object": "sse_transition_network",
        "format": "gpickle.gz",
        "n_rows": None,
        "n_columns": None,
        "n_nodes": sse_network.number_of_nodes(),
        "n_edges": sse_network.number_of_edges(),
        "path": str(network_path.relative_to(PROJECT_ROOT)),
    }
)

export_manifest = pd.DataFrame(saved_outputs)
export_manifest.to_parquet(OUTPUT_DIR / "export_manifest.parquet", index=False)
display(export_manifest)